# 02 - Train the QuantumSentinel ML detector on backend data

Trains the logistic-regression eavesdropper detector on sessions generated by the backend (`engine='aer'` = real Qiskit circuits). Saves `backend/model.json`, which the API and the frontend load at startup.

In [ ]:
import sys, json
sys.path.insert(0, '../backend')
import main   # FastAPI backend module (also exposes generate_dataset)

# 'aer' = train on real Qiskit circuits (slow but authentic).
# 'classical' = NumPy twin (fast; statistically identical).
ENGINE = 'aer'
X, y = main.generate_dataset(6000, seed=42, engine=ENGINE, n=1000)
X_test, y_test = main.generate_dataset(3000, seed=7, engine=ENGINE, n=1000)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler().fit(X)
clf = LogisticRegression(max_iter=2000).fit(scaler.transform(X), y)

prob = clf.predict_proba(scaler.transform(X_test))[:, 1]
pred = prob > 0.5
acc = (pred == y_test).mean()
det = pred[y_test == 1].mean()      # detection rate
fa  = pred[y_test == 0].mean()      # false-alarm rate
print(f'accuracy={acc:.3f}  detection={det:.3f}  false-alarm={fa:.3f}')


In [ ]:
model = {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
         "coef": clf.coef_[0].tolist(), "b": float(clf.intercept_[0]),
         "metrics": {"accuracy": acc, "detection": det, "false_alarm": fa,
                     "n_train": len(X), "n_test": len(X_test), "engine_note": f'trained on engine={ENGINE}'}}
with open('../backend/model.json', 'w') as fh:
    json.dump(model, fh, indent=2)
print('saved -> backend/model.json  (restart uvicorn to load it)')
